# Sacred Fruit: reentrenamiento del modelo

Entorno de ejecución: **GPU T4**. Ejecutá todo en orden y pegá tu Private API Key de Roboflow cuando la pida.

- Las copias de `best.pt`, `last.pt` y los gráficos se guardan en Google Drive (`MyDrive/sacred_fruit`). Si la sesión se corta, volvé a correr todo: el entrenamiento sigue desde el último `last.pt` guardado.
- `EXTRA_DATASETS` tiene los datasets públicos que se suman al tuyo. Los nuevos (defectos, sobrepasados, banana, otras frutas) se agregan acá cuando estén aprobados.

In [ ]:
!nvidia-smi -L
!pip -q install ultralytics roboflow imagehash

In [ ]:
import getpass
ROBOFLOW_API_KEY = globals().get("ROBOFLOW_API_KEY") or getpass.getpass("Pegá tu Private API Key de Roboflow y apretá Enter: ")
WORKSPACE = "dario-avalos-reckziegel"
PROJECT = "sacred-fruit"
VERSION = None
EXTRA_DATASETS = [
    ("yashas", "orange_ripeness", {"ripe_orange": "naranja_madura", "unripe_orange": "naranja_verde"}),
    ("yashas", "mango_ripeness", {"ripe_mango": "mango_maduro", "raw_mango": "mango_verde"}),
    ("apu-assignment", "tomato-ripeness-segmentation", {"fully_ripe": "tomate_maduro", "turning": "tomate_verde", "unripe": "tomate_verde"}),
]
BASE_MODEL = "yolo11s-seg.pt"
EPOCHS = None
IMGSZ = 640
BG_RATIO = 0.15
MAX_PER_CLASS = 1500
DRIVE_DIR = "/content/drive/MyDrive/sacred_fruit"

In [ ]:
import os
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Sin Google Drive, las copias quedan solo en /content:', e)
    DRIVE_DIR = '/content/copias'
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
print('Copias en', DRIVE_DIR)

In [ ]:
import os, re, shutil, random, glob, json, zipfile, urllib.request
from pathlib import Path
from collections import Counter, defaultdict
from roboflow import Roboflow

random.seed(42)
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
versions = sorted(v.version for v in project.versions())
ver = VERSION or versions[-1]
print("Versiones disponibles:", versions, "-> uso", ver)
ds = project.version(int(str(ver).split('/')[-1])).download("yolov8", location="/content/rf_main", overwrite=True)
sources = [(Path(ds.location), None)]
for ws, pj, mapping in EXTRA_DATASETS:
    try:
        p = rf.workspace(ws).project(pj)
        v = max(int(str(x.version).split('/')[-1]) for x in p.versions())
        d = p.version(v).download("yolov8", location=f"/content/rf_{pj}", overwrite=False)
        sources.append((Path(d.location), mapping))
    except Exception as e:
        print("No se pudo bajar", pj, "->", e)

In [ ]:
import yaml
CLASSES = ['banana_madura','banana_sobrepasada','banana_verde','defecto','mango_maduro','mango_sobrepasado','mango_verde',
           'naranja_madura','naranja_sobrepasada','naranja_verde','tomate_maduro','tomate_sobrepasado','tomate_verde']
IDX = {c:i for i,c in enumerate(CLASSES)}

def norm(name):
    n = name.strip().lower().replace(' ','_').replace('-','_')
    n = n.replace('maduro','madur_').replace('madura','madur_').replace('sobrepasado','sobrepasad_').replace('sobrepasada','sobrepasad_')
    for c in CLASSES:
        k = c.replace('maduro','madur_').replace('madura','madur_').replace('sobrepasado','sobrepasad_').replace('sobrepasada','sobrepasad_')
        if n == k: return c
    return None

items = []
for src, mapping in sources:
    names = yaml.safe_load(open(src/'data.yaml'))['names']
    names = names if isinstance(names, list) else [names[k] for k in sorted(names)]
    key = lambda n: n.strip().lower().replace(' ','_').replace('-','_')
    remap = {i: (mapping.get(key(n)) if mapping else norm(n)) for i, n in enumerate(names)}
    print(src.name, '->', {names[i]: v for i, v in remap.items()})
    for img in glob.glob(str(src/'*'/'images'/'*')):
        lbl = Path(img.replace('/images/','/labels/')).with_suffix('.txt')
        lines, unknown = [], False
        if lbl.exists():
            for ln in open(lbl).read().splitlines():
                p = ln.split()
                if not p: continue
                c = remap.get(int(p[0]))
                if c is None: unknown = True
                elif len(p) >= 7: lines.append(' '.join([str(IDX[c])] + p[1:]))
        if unknown or (mapping and not lines): continue
        stem = re.split(r'_(jpg|jpeg|png|webp)\.rf\.', Path(img).name, flags=re.I)[0]
        items.append(dict(img=img, lines=lines, group=f"{src.name}/{stem}"))

groups = defaultdict(list)
for it in items: groups[it['group']].append(it)
print(f"{len(items)} imágenes, {len(groups)} fotos originales (el resto son aumentaciones de Roboflow)")
cnt = Counter(CLASSES[int(l.split()[0])] for it in items for l in it['lines'])
for c in CLASSES: print(f"  {c:22s} {cnt[c]}")

In [ ]:
# Submuestreo: las clases con muchas etiquetas (mango maduro, tomate verde) tapan a las chicas.
# Se descartan fotos originales enteras, solo si todas sus clases ya pasaron MAX_PER_CLASS.
keys = list(groups); random.shuffle(keys)
kept, used = {}, Counter()
for k in keys:
    its = groups[k]
    cls = Counter(CLASSES[int(l.split()[0])] for it in its for l in it['lines'])
    if cls and all(used[c] >= MAX_PER_CLASS for c in cls):
        continue
    kept[k] = its; used.update(cls)
print(f"Fotos originales: {len(groups)} -> {len(kept)}")
for c in CLASSES: print(f"  {c:22s} {cnt[c]:6d} -> {used[c]}")
groups = kept

In [ ]:
keys = list(groups)
nval = max(1, int(len(keys)*0.2))
split = {k: ('val' if i < nval else 'train') for i, k in enumerate(keys)}

OUT = Path('/content/data'); shutil.rmtree(OUT, ignore_errors=True)
for s in ['train','val','neg_test']:
    (OUT/s/'images').mkdir(parents=True); (OUT/s/'labels').mkdir(parents=True)

def put(it, s, i):
    ext = Path(it['img']).suffix
    shutil.copy(it['img'], OUT/s/'images'/f"{i:05d}{ext}")
    open(OUT/s/'labels'/f"{i:05d}.txt",'w').write('\n'.join(it['lines']))

i = 0
for k, its in groups.items():
    for it in (its if split[k]=='train' else its[:1]):
        put(it, split[k], i); i += 1
ntrain = len(os.listdir(OUT/'train'/'images'))
print('train', ntrain, 'val', len(os.listdir(OUT/'val'/'images')))

In [ ]:
# Fondos sin fruta sacados de COCO val2017: enseñan al modelo a no ver frutas en techos, paredes, cajas, etc.
if not Path('/content/coco/val2017').exists():
    os.makedirs('/content/coco', exist_ok=True)
    urllib.request.urlretrieve('http://images.cocodataset.org/zips/val2017.zip', '/content/coco/val.zip')
    urllib.request.urlretrieve('http://images.cocodataset.org/annotations/annotations_trainval2017.zip', '/content/coco/ann.zip')
    zipfile.ZipFile('/content/coco/val.zip').extractall('/content/coco')
    zipfile.ZipFile('/content/coco/ann.zip').extractall('/content/coco')
ann = json.load(open('/content/coco/annotations/instances_val2017.json'))
# Solo se excluyen las fotos con banana o naranja. Manzanas, brócoli, zanahorias y otra comida
# quedan como fondo, y van primero para que el modelo aprenda que eso no es fruta del proyecto.
cat = {c['id']: c['name'] for c in ann['categories']}
bad = {a['image_id'] for a in ann['annotations'] if cat[a['category_id']] in ('banana', 'orange')}
food = {a['image_id'] for a in ann['annotations'] if cat[a['category_id']] in ('apple', 'broccoli', 'carrot', 'sandwich', 'pizza', 'donut', 'cake', 'hot dog')}
bg = [im['file_name'] for im in ann['images'] if im['id'] not in bad]
random.shuffle(bg)
bg.sort(key=lambda f: int(f.split('.')[0]) not in food)
print('fondos con otra comida:', sum(int(f.split('.')[0]) in food for f in bg))
n_bg = int(ntrain * BG_RATIO / (1 - BG_RATIO))
for j, f in enumerate(bg[:n_bg]):
    shutil.copy(f'/content/coco/val2017/{f}', OUT/'train'/'images'/f'bg_{j:05d}.jpg')
    open(OUT/'train'/'labels'/f'bg_{j:05d}.txt','w').close()
for j, f in enumerate(bg[n_bg:n_bg+300]):
    shutil.copy(f'/content/coco/val2017/{f}', OUT/'neg_test'/'images'/f'neg_{j:05d}.jpg')
print('fondos agregados al train:', n_bg, '| negativos de prueba: 300')

yaml.safe_dump({'path': str(OUT), 'train': 'train/images', 'val': 'val/images', 'names': {i:c for i,c in enumerate(CLASSES)}},
               open(OUT/'data.yaml','w'), allow_unicode=True, sort_keys=False)

In [ ]:
from ultralytics import YOLO
import glob, shutil

if EPOCHS is None:
    EPOCHS = 100 if ntrain < 3000 else 80 if ntrain < 6000 else 60
print('Épocas:', EPOCHS, '| imágenes de entrenamiento:', ntrain)

RUN = Path('/content/runs/sacred')
saved_last = Path(DRIVE_DIR) / 'last.pt'

def backup(trainer):
    if trainer.epoch % 5 == 0 or trainer.epoch == trainer.epochs - 1:
        for name in ('last.pt', 'best.pt'):
            f = RUN / 'weights' / name
            if f.exists(): shutil.copy(f, Path(DRIVE_DIR) / name)
        if (RUN / 'results.csv').exists(): shutil.copy(RUN / 'results.csv', Path(DRIVE_DIR) / 'results.csv')

if saved_last.exists() and not (RUN / 'weights' / 'last.pt').exists():
    (RUN / 'weights').mkdir(parents=True, exist_ok=True)
    shutil.copy(saved_last, RUN / 'weights' / 'last.pt')

if (RUN / 'weights' / 'last.pt').exists():
    print('Retomo desde last.pt')
    model = YOLO(str(RUN / 'weights' / 'last.pt'))
    model.add_callback('on_fit_epoch_end', backup)
    model.train(resume=True)
else:
    model = YOLO(BASE_MODEL)
    model.add_callback('on_fit_epoch_end', backup)
    model.train(data=str(OUT/'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ, batch=16, patience=20, cos_lr=True,
                hsv_h=0.01, hsv_s=0.5, hsv_v=0.4, degrees=15, scale=0.5, translate=0.1, fliplr=0.5, flipud=0.1,
                mosaic=1.0, mixup=0.1, copy_paste=0.3, close_mosaic=15, project='/content/runs', name='sacred', exist_ok=True)

In [ ]:
best = YOLO(str(RUN / 'weights' / 'best.pt'))
m = best.val(data=str(OUT/'data.yaml'), imgsz=IMGSZ, split='val', batch=8, plots=True, project='/content/runs', name='val', exist_ok=True)
print(f"mAP50 máscara: {m.seg.map50:.3f} | mAP50-95 máscara: {m.seg.map:.3f}\n")
for i, c in enumerate(CLASSES):
    try: print(f"  {c:22s} mAP50 {m.seg.class_result(list(m.seg.ap_class_index).index(i))[2]:.3f}")
    except ValueError: print(f"  {c:22s} sin ejemplos en val")

# Matriz de confusión: filas = clase predicha, columnas = clase real (la última es "fondo").
import numpy as np
cm = m.confusion_matrix.matrix.astype(int)
names = [c[:10] for c in CLASSES] + ['fondo']
print('\npred \\ real'.ljust(13) + ''.join(n.rjust(11) for n in names))
for i, row in enumerate(cm):
    print(names[i].ljust(13) + ''.join(str(v).rjust(11) for v in row))
for f in glob.glob('/content/runs/val/*.png'):
    shutil.copy(f, DRIVE_DIR)

In [ ]:
# Falsos positivos en fotos sin fruta. Una sola pasada con conf 0.25, de a una imagen para no llenar la GPU.
import torch
negs = sorted(glob.glob(str(OUT/'neg_test'/'images'/'*')))
top = []
for j, f in enumerate(negs):
    r = best.predict(f, conf=0.25, imgsz=IMGSZ, verbose=False)[0]
    top.append(float(r.boxes.conf.max()) if len(r.boxes) else 0.0)
    del r
    if j % 50 == 0: torch.cuda.empty_cache()
for conf in [0.25, 0.4, 0.5]:
    fp = sum(t >= conf for t in top)
    print(f"conf {conf}: {fp}/{len(negs)} imágenes sin fruta con alguna detección falsa ({100*fp/len(negs):.1f} %)")

In [ ]:
shutil.copy(RUN / 'weights' / 'best.pt', '/content/best.pt')
shutil.copy(RUN / 'weights' / 'best.pt', Path(DRIVE_DIR) / 'best.pt')
print('best.pt guardado en', DRIVE_DIR)
from google.colab import files
files.download('/content/best.pt')